In [1]:
import pandas as pd

In [7]:
water = pd.read_csv("../raw/area-wise-waterbodies-india-statewise.csv")
dim_state = pd.read_csv("../clean/dim_state.csv")

In [4]:
print(water.shape)
print(water.columns.tolist())
print(water.head(10).to_string(index=False))
print(water.dtypes)

(37, 8)
['State', '< 0.5 ha', '>= 0.5 - < 1 ha', '>= 1 - < 2 ha', '>= 2 - < 10 ha', '>= 10 - < 50 ha', '>= 50 - < 100 ha', '>= 100 ha']
                 State  < 0.5 ha  >= 0.5 - < 1 ha  >= 1 - < 2 ha  >= 2 - < 10 ha  >= 10 - < 50 ha  >= 50 - < 100 ha  >= 100 ha
     Andaman & Nicobar         9                1              2               9                4                 6          2
        Andhra Pradesh      3342             5265           7203           14535             5057               783        548
     Arunachal Pradesh       187              150            222             488              214                13          1
                 Assam      8796             2209           1416            1574              539                63         39
                 Bihar     36455            13572           6268            3964              929               110         72
          Chhattisgarh     50490            25155          17755            9669             1183     

In [5]:
print(water.isna().sum())

State               0
< 0.5 ha            0
>= 0.5 - < 1 ha     0
>= 1 - < 2 ha       0
>= 2 - < 10 ha      0
>= 10 - < 50 ha     0
>= 50 - < 100 ha    0
>= 100 ha           0
dtype: int64


In [8]:
water_states = set(water["State"].str.strip())
dim_states = set(dim_state["state_name"].str.strip())

print("In waterbody file but not dim_state:")
print(sorted(water_states - dim_states))

print("\nIn dim_state but not waterbody file:")
print(sorted(dim_states - water_states))

In waterbody file but not dim_state:
['Andaman & Nicobar', 'Dadara & Nagar Havelli', 'Daman & Diu', 'Jammu & Kashmir', 'Pondicherry']

In dim_state but not waterbody file:
['Andaman and Nicobar Islands', 'Dadra and Nagar Haveli and Daman and Diu', 'Jammu and Kashmir', 'Puducherry']


In [9]:
water_state_map = {
    "Andaman & Nicobar": "Andaman and Nicobar Islands",
    "Dadara & Nagar Havelli": "Dadra and Nagar Haveli and Daman and Diu",
    "Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Pondicherry": "Puducherry"
}

water["state_name"] = water["State"].str.strip().replace(water_state_map)

print(water["state_name"].unique())

<ArrowStringArray>
[             'Andaman and Nicobar Islands',
                           'Andhra Pradesh',
                        'Arunachal Pradesh',
                                    'Assam',
                                    'Bihar',
                             'Chhattisgarh',
                               'Chandigarh',
 'Dadra and Nagar Haveli and Daman and Diu',
                                    'Delhi',
                                      'Goa',
                                  'Gujarat',
                         'Himachal Pradesh',
                                  'Haryana',
                                'Jharkhand',
                        'Jammu and Kashmir',
                                'Karnataka',
                                   'Kerala',
                              'Lakshadweep',
                                   'Ladakh',
                              'Maharashtra',
                                'Meghalaya',
                                  'M

In [10]:
print(
    water[
        water["state_name"] ==
        "Dadra and Nagar Haveli and Daman and Diu"
    ].to_string(index=False)
)

                 State  < 0.5 ha  >= 0.5 - < 1 ha  >= 1 - < 2 ha  >= 2 - < 10 ha  >= 10 - < 50 ha  >= 50 - < 100 ha  >= 100 ha                               state_name
           Daman & Diu         3                1              3              19                3                 0          0 Dadra and Nagar Haveli and Daman and Diu
Dadara & Nagar Havelli         0                1              0               2                0                 0          1 Dadra and Nagar Haveli and Daman and Diu


In [11]:
size_cols = [
    "< 0.5 ha",
    ">= 0.5 - < 1 ha",
    ">= 1 - < 2 ha",
    ">= 2 - < 10 ha",
    ">= 10 - < 50 ha",
    ">= 50 - < 100 ha",
    ">= 100 ha"
]

water = (
    water
    .groupby("state_name", as_index=False)[size_cols]
    .sum()
)

print(water.shape)
print(water[water["state_name"] == 
            "Dadra and Nagar Haveli and Daman and Diu"])

(36, 8)
                                 state_name  < 0.5 ha  >= 0.5 - < 1 ha  \
7  Dadra and Nagar Haveli and Daman and Diu         3                2   

   >= 1 - < 2 ha  >= 2 - < 10 ha  >= 10 - < 50 ha  >= 50 - < 100 ha  >= 100 ha  
7              3              21                3                 0          1  


In [13]:
water["waterbody_count_total"] = water[size_cols].sum(axis=1)

print(water[["state_name", "waterbody_count_total"]].head(40))

                                  state_name  waterbody_count_total
0                Andaman and Nicobar Islands                     33
1                             Andhra Pradesh                  36733
2                          Arunachal Pradesh                   1275
3                                      Assam                  14636
4                                      Bihar                  61370
5                                 Chandigarh                      1
6                               Chhattisgarh                 104447
7   Dadra and Nagar Haveli and Daman and Diu                     33
8                                      Delhi                     53
9                                        Goa                    391
10                                   Gujarat                  16273
11                                   Haryana                   5726
12                          Himachal Pradesh                    217
13                         Jammu and Kashmir    

In [15]:
print("Unique states:", water["state_name"].nunique())
print("Missing values:")
print(water.isna().sum())

print("\nNegative values:")
print((water[size_cols] < 0).sum().sum())

print("\nZero-total states:")
print(
    water.loc[
        water["waterbody_count_total"] == 0,
        "state_name"
    ].tolist()
)

Unique states: 36
Missing values:
state_name               0
< 0.5 ha                 0
>= 0.5 - < 1 ha          0
>= 1 - < 2 ha            0
>= 2 - < 10 ha           0
>= 10 - < 50 ha          0
>= 50 - < 100 ha         0
>= 100 ha                0
waterbody_count_total    0
dtype: int64

Negative values:
0

Zero-total states:
[]


In [17]:
water = water.merge(
    dim_state[["state_id", "state_name"]],
    on="state_name",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(water))
print("Missing state_id:", water["state_id"].isna().sum())

print(
    water[
        water["state_id"].isna()
    ][["state_name"]]
)

Rows: 36
Missing state_id: 0
Empty DataFrame
Columns: [state_name]
Index: []


In [19]:
for col in size_cols:
    share_col = col.replace(" ", "_").replace("<", "lt").replace(">=", "gte")
    water[share_col + "_pct"] = (
        water[col] / water["waterbody_count_total"] * 100
    )

print(water.head().to_string(index=False))

                 state_name  < 0.5 ha  >= 0.5 - < 1 ha  >= 1 - < 2 ha  >= 2 - < 10 ha  >= 10 - < 50 ha  >= 50 - < 100 ha  >= 100 ha  waterbody_count_total state_id  lt_0.5_ha_pct  gte_0.5_-_lt_1_ha_pct  gte_1_-_lt_2_ha_pct  gte_2_-_lt_10_ha_pct  gte_10_-_lt_50_ha_pct  gte_50_-_lt_100_ha_pct  gte_100_ha_pct
Andaman and Nicobar Islands         9                1              2               9                4                 6          2                     33    IN-AN      27.272727               3.030303             6.060606             27.272727              12.121212               18.181818        6.060606
             Andhra Pradesh      3342             5265           7203           14535             5057               783        548                  36733    IN-AP       9.098086              14.333161            19.609071             39.569325              13.766913                2.131598        1.491847
          Arunachal Pradesh       187              150            222       

In [21]:
water_rename = {
    "< 0.5 ha": "waterbody_count_lt_0_5ha",
    ">= 0.5 - < 1 ha": "waterbody_count_0_5_1ha",
    ">= 1 - < 2 ha": "waterbody_count_1_2ha",
    ">= 2 - < 10 ha": "waterbody_count_2_10ha",
    ">= 10 - < 50 ha": "waterbody_count_10_50ha",
    ">= 50 - < 100 ha": "waterbody_count_50_100ha",
    ">= 100 ha": "waterbody_count_gte_100ha"
}

water_clean = water.rename(columns=water_rename)

share_rename = {
    "waterbody_count_lt_0_5ha": "waterbody_share_lt_0_5ha_pct",
    "waterbody_count_0_5_1ha": "waterbody_share_0_5_1ha_pct",
    "waterbody_count_1_2ha": "waterbody_share_1_2ha_pct",
    "waterbody_count_2_10ha": "waterbody_share_2_10ha_pct",
    "waterbody_count_10_50ha": "waterbody_share_10_50ha_pct",
    "waterbody_count_50_100ha": "waterbody_share_50_100ha_pct",
    "waterbody_count_gte_100ha": "waterbody_share_gte_100ha_pct"
}

In [22]:
# Rename original count columns
water_clean = water_clean.rename(columns=water_rename)

count_cols = list(water_rename.values())

# Create percentage-share columns
for col in count_cols:
    share_col = col.replace("waterbody_count_", "waterbody_share_") + "_pct"
    water_clean[share_col] = (
        water_clean[col] /
        water_clean["waterbody_count_total"] * 100
    )

# Keep only the final analytical columns
water_clean = water_clean[
    ["state_id", "state_name",
     "waterbody_count_total"] +
    count_cols +
    [c for c in water_clean.columns
     if c.startswith("waterbody_share_")]
]

print(water_clean.shape)
print(water_clean.columns.tolist())

(36, 17)
['state_id', 'state_name', 'waterbody_count_total', 'waterbody_count_lt_0_5ha', 'waterbody_count_0_5_1ha', 'waterbody_count_1_2ha', 'waterbody_count_2_10ha', 'waterbody_count_10_50ha', 'waterbody_count_50_100ha', 'waterbody_count_gte_100ha', 'waterbody_share_lt_0_5ha_pct', 'waterbody_share_0_5_1ha_pct', 'waterbody_share_1_2ha_pct', 'waterbody_share_2_10ha_pct', 'waterbody_share_10_50ha_pct', 'waterbody_share_50_100ha_pct', 'waterbody_share_gte_100ha_pct']


In [23]:
share_cols = [
    c for c in water_clean.columns
    if c.startswith("waterbody_share_")
]

print(
    water_clean[share_cols].sum(axis=1).describe()
)

count    3.600000e+01
mean     1.000000e+02
std      1.100768e-14
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
dtype: float64


In [24]:
share_cols = [
    c for c in water_clean.columns
    if c.startswith("waterbody_share_")
]

water_clean = water_clean.drop(columns=share_cols)

print(water_clean.columns.tolist())
print(water_clean.shape)

['state_id', 'state_name', 'waterbody_count_total', 'waterbody_count_lt_0_5ha', 'waterbody_count_0_5_1ha', 'waterbody_count_1_2ha', 'waterbody_count_2_10ha', 'waterbody_count_10_50ha', 'waterbody_count_50_100ha', 'waterbody_count_gte_100ha']
(36, 10)


In [27]:
water_clean.to_csv(
    "../clean/fact_state_waterbodies.csv",
    index=False,
    encoding="utf-8"
)

print("Saved:", "../clean/fact_state_waterbodies.csv")
print("Shape:", water_clean.shape)

Saved: ../clean/fact_state_waterbodies.csv
Shape: (36, 10)
